Latest Version

In [ ]:
def letterbox(frame, target_w, target_h):
    h, w = frame.shape[:2]
    scale = min(target_w / w, target_h / h)

    new_w = int(w * scale)
    new_h = int(h * scale)

    resized = cv2.resize(frame, (new_w, new_h), interpolation=cv2.INTER_LINEAR)

    canvas = np.zeros((target_h, target_w, 3), dtype=np.uint8)

    x_offset = (target_w - new_w) // 2
    y_offset = (target_h - new_h) // 2

    canvas[y_offset:y_offset+new_h, x_offset:x_offset+new_w] = resized
    return canvas

In [2]:
import cv2
import numpy as np
import pyaudio
import math
import time
from collections import deque, Counter

# ==========================================
# 1. PURE NUMPY AUDIO ENGINE (Crash-Proof)
# ==========================================
class AudioEngine:
    def __init__(self):
        self.SAMPLE_RATE = 44100
        self.p = pyaudio.PyAudio()
        
        # Audio Targets (Incoming from Vision)
        self.params = {
            "freq": 440.0, 
            "filter": 0.0, 
            "lfo_rate": 4.0, 
            "volume": 0.0, 
            "pan": 0.5
        }
        
        # Internal Smoothed Values (Current State)
        self.curr = {
            "freq": 440.0, 
            "filter": 0.0, 
            "lfo": 4.0, 
            "vol": 0.0, 
            "pan": 0.5
        }
        
        self.phase = 0.0
        self.lfo_phase = 0.0
        
        # Stereo Reverb Buffers
        self.delay_len_L = int(self.SAMPLE_RATE * 0.18) 
        self.delay_len_R = int(self.SAMPLE_RATE * 0.22)
        
        self.buff_L = np.zeros(self.delay_len_L, dtype=np.float32)
        self.buff_R = np.zeros(self.delay_len_R, dtype=np.float32)
        self.ptr_L = 0
        self.ptr_R = 0
        
        self.stream = self.p.open(
            format=pyaudio.paFloat32, channels=2, rate=self.SAMPLE_RATE, output=True, stream_callback=self.callback
        )

    def update(self, freq, filter_val, lfo_rate, vol, pan):
        self.params["freq"] = freq
        self.params["filter"] = filter_val
        self.params["lfo_rate"] = lfo_rate
        self.params["volume"] = vol
        self.params["pan"] = pan

    def callback(self, in_data, frame_count, time_info, status):
        # 1. Update & Smooth Parameters
        # Frequency snaps instantly (No smoothing)
        self.curr["freq"] = self.params["freq"] 
        
        # Smooth the rest explicitly (FIXED THE KEY ERROR HERE)
        self.curr["filter"] += (self.params["filter"] - self.curr["filter"]) * 0.1
        self.curr["lfo"]    += (self.params["lfo_rate"] - self.curr["lfo"]) * 0.1
        self.curr["vol"]    += (self.params["volume"]   - self.curr["vol"]) * 0.1
        self.curr["pan"]    += (self.params["pan"]      - self.curr["pan"]) * 0.1

        # Optimization: Return silence if volume is 0
        if self.curr["vol"] < 0.001 and np.max(np.abs(self.buff_L)) < 0.001:
            return (np.zeros(frame_count * 2, dtype=np.float32), pyaudio.paContinue)

        t = np.arange(frame_count, dtype=np.float32)

        # 2. LFO Generation
        lfo_val = np.sin(self.lfo_phase + t * 2 * np.pi * self.curr["lfo"] / self.SAMPLE_RATE) * 8.0
        self.lfo_phase += frame_count * 2 * np.pi * self.curr["lfo"] / self.SAMPLE_RATE
        
        # 3. Oscillator Generation
        mod_freq = self.curr["freq"] + lfo_val
        phase_inc = 2 * np.pi * mod_freq / self.SAMPLE_RATE
        phases = self.phase + np.cumsum(phase_inc)
        self.phase = phases[-1] % (2 * np.pi)
        
        sine = np.sin(phases)
        saw = sine + 0.5*np.sin(2*phases) + 0.25*np.sin(4*phases)
        
        # 4. Filter Mix (Crossfade)
        mono_sig = (sine * self.curr["filter"]) + (saw * (1.0 - self.curr["filter"]))
        mono_sig *= self.curr["vol"] * 0.2 

        # 5. Stereo Panning
        pan_rad = self.curr["pan"] * (np.pi / 2.0)
        sig_L = mono_sig * np.cos(pan_rad)
        sig_R = mono_sig * np.sin(pan_rad)

        # 6. Reverb (Vectorized Ring Buffers)
        # Left Channel
        indices_L = (np.arange(frame_count) + self.ptr_L) % self.delay_len_L
        delay_L = self.buff_L[indices_L]
        out_L = sig_L + (delay_L * 0.4) 
        self.buff_L[indices_L] = sig_L + (delay_L * 0.6) 
        self.ptr_L = (self.ptr_L + frame_count) % self.delay_len_L
        
        # Right Channel
        indices_R = (np.arange(frame_count) + self.ptr_R) % self.delay_len_R
        delay_R = self.buff_R[indices_R]
        out_R = sig_R + (delay_R * 0.4)
        self.buff_R[indices_R] = sig_R + (delay_R * 0.6)
        self.ptr_R = (self.ptr_R + frame_count) % self.delay_len_R
        
        # 7. Interleave
        stereo_out = np.empty(frame_count * 2, dtype=np.float32)
        stereo_out[0::2] = out_L
        stereo_out[1::2] = out_R
        
        return (stereo_out, pyaudio.paContinue)

    def close(self):
        self.stream.stop_stream(); self.stream.close(); self.p.terminate()

# ==========================================
# 2. VISION APP
# ==========================================
class VisionHarpFinal:
    def __init__(self):
        self.cap = cv2.VideoCapture(0)
        self.audio = AudioEngine()
        
        self.kf = cv2.KalmanFilter(4, 2)
        self.kf.measurementMatrix = np.array([[1,0,0,0],[0,1,0,0]], np.float32)
        self.kf.transitionMatrix = np.array([[1,0,1,0],[0,1,0,1],[0,0,1,0],[0,0,0,1]], np.float32)
        self.kf.processNoiseCov = np.eye(4, dtype=np.float32) * 0.03
        self.kf.statePost = np.array([0, 0, 0, 0], np.float32)
        
        self.lower_skin = None; self.upper_skin = None; self.is_calibrated = False
        self.finger_buffer = deque(maxlen=5)
        self.particles = deque(maxlen=20) 
        
        self.scale_mode = "Major"
        self.base_root = 130.81
        self.scale_data = {
            "Major": {"intervals": [0, 2, 4, 5, 7, 9, 11], "names": ["C", "D", "E", "F", "G", "A", "B"]},
            "Minor": {"intervals": [0, 2, 3, 5, 7, 8, 10], "names": ["C", "D", "Eb", "F", "G", "Ab", "Bb"]},
            "Blues": {"intervals": [0, 3, 5, 6, 7, 10, 12], "names": ["C", "Eb", "F", "F#", "G", "Bb", "C"]},
            "Hirajoshi": {"intervals": [0, 2, 3, 7, 8], "names": ["C", "D", "Eb", "G", "Ab"]}
        }
        self.current_scale = []; self.current_note_names = []
        self.update_scale()

    def update_scale(self):
        data = self.scale_data[self.scale_mode]
        pattern = data["intervals"]
        names = data["names"]
        full_scale = []
        full_names = []
        for octave in [0.5, 1.0, 2.0]:
            for i, semitone in enumerate(pattern):
                freq = (self.base_root * octave) * (2 ** (semitone / 12.0))
                full_scale.append(freq)
                oct_suffix = "2" if octave == 0.5 else ("3" if octave == 1.0 else "4")
                name_idx = i % len(names)
                full_names.append(f"{names[name_idx]}{oct_suffix}")
        self.current_scale = full_scale
        self.current_note_names = full_names

    def calibrate(self, frame, rect):
        x, y, w, h = rect
        roi = frame[y:y+h, x:x+w]
        hsv = cv2.cvtColor(roi, cv2.COLOR_BGR2HSV)
        mean = np.mean(hsv, axis=(0, 1))
        self.lower_skin = np.array([max(0, mean[0]-25), 30, 40], dtype=np.uint8)
        self.upper_skin = np.array([min(180, mean[0]+25), 255, 255], dtype=np.uint8)
        self.is_calibrated = True
        print(f"Calibrated: {mean}")

    def get_palm_center(self, mask_roi):
        dist_transform = cv2.distanceTransform(mask_roi, cv2.DIST_L2, 5)
        _, max_val, _, max_loc = cv2.minMaxLoc(dist_transform)
        return max_loc, max_val

    def robust_finger_count(self, contour, palm_center, palm_radius):
        if contour is None or len(contour) < 5: return 0
        area = cv2.contourArea(contour)
        perim = cv2.arcLength(contour, True)
        if perim == 0: return 0
        circularity = 4 * math.pi * (area / (perim ** 2))
        if circularity > 0.65: return 0 

        hull = cv2.convexHull(contour, returnPoints=False)
        if hull is None or len(hull) <= 2: return 0
        try: defects = cv2.convexityDefects(contour, hull)
        except: return 0
        count = 0
        if defects is not None:
            for i in range(defects.shape[0]):
                s, e, f, d = defects[i, 0]
                start = tuple(contour[s][0])
                depth = d / 256.0
                if depth > palm_radius * 0.3:
                    if start[1] < palm_center[1] + (palm_radius * 0.9):
                        if math.hypot(start[0]-palm_center[0], start[1]-palm_center[1]) > palm_radius * 1.3:
                            count += 1
        return min(count + 1 if count > 0 else 0, 5)

    def draw_particles(self, frame, sx, sy, color):
        self.particles.append((sx, sy))
        total_p = len(self.particles)
        for i in range(1, total_p):
            calc_thick = int((i / total_p) * 5)
            thickness = max(1, calc_thick) 
            cv2.line(frame, self.particles[i-1], self.particles[i], color, thickness)

    def draw_grid_and_notes(self, frame, H, W, active_idx):
        num_notes = len(self.current_scale)
        col_w = W / num_notes
        for i in range(num_notes):
            x_pos = int(i * col_w)
            if i == active_idx:
                line_color = (0, 255, 255); text_color = (0, 255, 255); thickness = 2
            else:
                line_color = (40, 40, 40); text_color = (100, 100, 100); thickness = 1
            cv2.line(frame, (x_pos, 0), (x_pos, H), line_color, thickness)
            y_pos = 30 if i % 2 == 0 else 50
            note_name = self.current_note_names[i]
            cv2.putText(frame, note_name, (x_pos + 5, y_pos), cv2.FONT_HERSHEY_SIMPLEX, 0.4, text_color, 1)

    def run(self):
        cv2.namedWindow("VisionHarp Final", cv2.WINDOW_NORMAL)
        while True:
            ret, frame = self.cap.read()
            if not ret: break
            frame = cv2.flip(frame, 1)
            H, W, _ = frame.shape
            
            cv2.putText(frame, f"SCALE: {self.scale_mode}", (10, H-40), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
            cv2.putText(frame, "[1]Maj [2]Min [3]Blues [4]Hirajoshi", (10, H-15), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (150, 150, 150), 1)

            if not self.is_calibrated:
                cx, cy = W//2, H//2
                cv2.rectangle(frame, (cx-50, cy-50), (cx+50, cy+50), (0, 255, 0), 2)
                cv2.putText(frame, "PALM IN BOX -> PRESS 'C'", (cx-150, cy-70), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
                key = cv2.waitKey(1) & 0xFF
                if key == ord('c'): self.calibrate(frame, (cx-50, cy-50, 100, 100))
                if key == ord('q'): break
            else:
                hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
                mask = cv2.inRange(hsv, self.lower_skin, self.upper_skin)
                mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, np.ones((3,3),np.uint8))
                mask = cv2.dilate(mask, np.ones((3,3),np.uint8), iterations=3)
                mask = cv2.GaussianBlur(mask, (5, 5), 0)
                
                cnts, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
                found = False; active_note_idx = -1
                
                if cnts:
                    c = max(cnts, key=cv2.contourArea)
                    if cv2.contourArea(c) > 2000:
                        found = True
                        x, y, w, h = cv2.boundingRect(c)
                        roi = mask[y:y+h, x:x+w]
                        local_palm, palm_radius = self.get_palm_center(roi)
                        gx, gy = x + local_palm[0], y + local_palm[1]
                        
                        self.kf.predict()
                        meas = np.array([[np.float32(gx)], [np.float32(gy)]])
                        self.kf.correct(meas)
                        sx, sy = int(self.kf.statePost[0]), int(self.kf.statePost[1])
                        sx, sy = max(0, min(W, sx)), max(0, min(H, sy))
                        
                        safe_radius = max(1, min(200, int(palm_radius)))
                        raw_fingers = self.robust_finger_count(c, (gx, gy), palm_radius)
                        self.finger_buffer.append(raw_fingers)
                        stable_fingers = Counter(self.finger_buffer).most_common(1)[0][0]
                        
                        norm_x = sx / W
                        active_note_idx = int(norm_x * len(self.current_scale))
                        active_note_idx = max(0, min(active_note_idx, len(self.current_scale)-1))
                        target_freq = self.current_scale[active_note_idx]
                        current_note_name = self.current_note_names[active_note_idx]
                        
                        target_filt = 1.0 - (min(stable_fingers, 5) / 5.0)
                        norm_z = np.clip((safe_radius - 10) / 50.0, 0, 1.0)
                        target_lfo = 2 ** (2.0 + (norm_z * 6.0))
                        target_pan = np.clip(norm_x, 0.0, 1.0)
                        
                        self.audio.update(target_freq, target_filt, target_lfo, 1.0, target_pan)
                        
                        color = (int(50*target_filt), 255, int(255*(1-target_filt)))
                        self.draw_particles(frame, sx, sy, color)
                        cv2.circle(frame, (sx, sy), safe_radius, color, 2)
                        cv2.putText(frame, f"{current_note_name}", (sx+15, sy), cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 2)
                        
                self.draw_grid_and_notes(frame, H, W, active_note_idx)
                if not found:
                    self.audio.update(440, 0, 4, 0.0, 0.5)
                    self.particles.clear()

                key = cv2.waitKey(1) & 0xFF
                if key == ord('q'): break
                if key == ord('r'): self.is_calibrated = False
                if key == ord('1'): self.scale_mode = "Major"; self.update_scale()
                if key == ord('2'): self.scale_mode = "Minor"; self.update_scale()
                if key == ord('3'): self.scale_mode = "Blues"; self.update_scale()
                if key == ord('4'): self.scale_mode = "Hirajoshi"; self.update_scale()

            cv2.imshow("VisionHarp Final", frame)

        self.audio.close()
        self.cap.release()
        cv2.destroyAllWindows()

if __name__ == "__main__":
    app = VisionHarpFinal()
    app.run()

Calibrated: [ 19.9309  47.6629 183.4432]
Calibrated: [110.0312  67.8789 113.4458]
Calibrated: [110.6087  57.6047 136.8706]
